# 09 - Model Routing

## Scenario: Cost-Efficient Ticket Triage

Using a frontier model (like GPT-4o or Claude 3.5 Sonnet) for every single user request is a waste of money. 
If a user just asks "How do I reset my password?", a cheap, fast model (like GPT-4o-mini or Haiku) can easily handle it.

**Model Routing** is the architectural pattern where a cheap LLM acts as the gatekeeper. It evaluates the complexity of the request and routes it to an expensive model *only* if required. In this notebook, we build a Router for Northstar Support.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Defining the Router

In [2]:
from pydantic import BaseModel

class RouteDecision(BaseModel):
    complexity: str # "LOW" or "HIGH"
    reasoning: str

def get_routing_decision(user_query: str) -> RouteDecision:
    """Uses a CHEAP model to classify the request."""
    print("🚦 [Router] Analyzing ticket complexity using gpt-4o-mini (Cheap Model)...")
    
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini", # The cheap, fast model
            messages=[
                {"role": "system", "content": "Classify the support ticket. If it requires log analysis, coding, or deep reasoning, output HIGH. If it's a simple FAQ, output LOW."},
                {"role": "user", "content": user_query}
            ],
            response_format=RouteDecision
        )
        return completion.choices[0].message.parsed
    except Exception:
        # Fallback for MockOpenAI
        is_complex = "log" in user_query.lower() or "latency" in user_query.lower()
        return RouteDecision(
            complexity="HIGH" if is_complex else "LOW",
            reasoning="Mock classification based on keywords."
        )


## 2. Executing the Route

In [3]:
def handle_support_ticket(user_query: str):
    print(f"\n📩 New Ticket: '{user_query}'")
    
    decision = get_routing_decision(user_query)
    
    if decision.complexity == "HIGH":
        print(f"🚀 [Router] Complexity is HIGH ({decision.reasoning}). Routing to GPT-4o (Expensive).")
        # In reality, execute the heavy agent here
        print("  🧠 [GPT-4o Agent] Performing deep log analysis...")
    else:
        print(f"⚡ [Router] Complexity is LOW ({decision.reasoning}). Handling with GPT-4o-mini (Cheap).")
        # Handle cheaply
        print("  🤖 [GPT-4o-mini] Here is the link to reset your password.")

# Simple Request
handle_support_ticket("How do I reset my account password?")

# Complex Request
handle_support_ticket("I'm seeing a massive latency spike when I query the checkout API and the logs show a deadlock.")



📩 New Ticket: 'How do I reset my account password?'
🚦 [Router] Analyzing ticket complexity using gpt-4o-mini (Cheap Model)...
⚡ [Router] Complexity is LOW (Mock classification based on keywords.). Handling with GPT-4o-mini (Cheap).
  🤖 [GPT-4o-mini] Here is the link to reset your password.

📩 New Ticket: 'I'm seeing a massive latency spike when I query the checkout API and the logs show a deadlock.'
🚦 [Router] Analyzing ticket complexity using gpt-4o-mini (Cheap Model)...
🚀 [Router] Complexity is HIGH (Mock classification based on keywords.). Routing to GPT-4o (Expensive).
  🧠 [GPT-4o Agent] Performing deep log analysis...


## Checkpoint

**1. What is the primary benefit of Model Routing?**
- A) It combines multiple models to generate one sentence.
- B) It prevents the system from overpaying for simple tasks by using cheap models as gatekeepers.
- C) It bypasses API rate limits entirely.
- D) It trains a new model from scratch on every request.
